In [2]:
import originpro as op 
import numpy as np 

In [13]:
def power_load(CT,omegaR=213.36,kappa=0.7,KP=1.0,sigma=0.1,Cd=0.01):
    ret=CT/omegaR*(CT**1.5/(2*np.sqrt(kappa))+KP*sigma*Cd/4.0)**-1
    return ret

def FM(CT,kappa,KP,sigma,Cd=0.01):
    ret=(CT**1.5/2)/(CT**1.5/2/np.sqrt(kappa)+KP*sigma*Cd/4.0)
    return ret 

op.attach()
wks=op.find_sheet("w","[Book1]Sheet1")


x=np.linspace(0,0.18,100)
sigma=0.1 
kappa=0.6
KP=0.7
CT=x*sigma
Cd=0.01

y1=power_load(CT=CT,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)
y2=FM(CT=CT,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)

wks.clear(0)
wks.from_list(0,x,lname=r"桨叶拉力系数 \q(CT/\sigma)")
wks.from_list(1,y1*1000,lname=r"功率载荷 \q(q/\rm{(N/kW)})")
wks.from_list(2,y2,lname=r"悬停效率 FM")

CT_max=(np.sqrt(kappa)*KP*sigma*Cd)**(2/3) 
FM_max=2/3*np.sqrt(kappa)
pl=power_load(CT=CT_max,sigma=sigma,KP=KP,kappa=kappa,Cd=Cd)
wks.from_list(3,CT_max/sigma,lname=r"最大 桨叶拉力系数 \q(CT/\sigma)")
wks.from_list(4,pl*1000,lname=r"最大 功率载荷 \q(q/\rm{(N/kW)})")
wks.from_list(5,FM_max,lname=r"最大 悬停效率 FM")

op.detach()

In [16]:
def _fun(FM,kappa,Kt,Kp,sigma):
    ret=1/FM 
    ret=ret-1/np.sqrt(kappa)
    ret=ret/(2.6*Kp/((kappa*Kt)**1.5*np.sqrt(sigma)))
    ret=1/ret 
    return ret

def fun(FM,kappa,sigma):
    # ret=1/FM 
    # ret=ret-kappa
    # ret=ret/(2.6/np.sqrt(sigma))
    # ret=1/ret 
    ret=(1/FM-kappa)*np.sqrt(sigma)/2.6
    return 1/ret

op.attach()
wks=op.find_sheet("w","[Book2]Sheet1")

x=np.linspace(1,1.4,100)
kappa=x

wks.clear()
wks.from_list(col=0,data=x,lname=r"\q(\kappa)")

FM=0.7
sigma=0.1
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=1,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.7
sigma=0.08
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=2,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.75
sigma=0.1
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=3,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.75
sigma=0.08
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=4,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.80
sigma=0.1
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=5,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.80
sigma=0.08
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=6,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.85
sigma=0.1
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=7,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

FM=0.85
sigma=0.08
y=fun(FM=FM,kappa=kappa,sigma=sigma)
y=y[:np.argmax(y)]
wks.from_list(col=8,data=y,lname=r"\q(FM={:.2f},\sigma={:.2f})".format(FM,sigma))

op.detach()

In [23]:
from NumOpt import Opti 

def func(PL,vinf):
    opti=Opti()
    vinf=np.linspace(0,500/3.6,100)
    vi=opti.variable(init_guess=np.ones_like(vinf))

    opti.subject_to([
        PL*9.8==2*1.225*vi*(vi+vinf)
    ])

    opti.ipopt_solver(verbose=False)
    sol=opti.solve()

    return sol(vi)


PL=60
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.clear()
wks.from_list(col=0,data=vinf,lname=r"\q(v_\infty)")
wks.from_list(col=1,data=vi,lname=r"\q(PL={:d})".format(PL))

PL=70
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.from_list(col=2,data=vi,lname=r"\q(PL={:d})".format(PL))

PL=80
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.from_list(col=3,data=vi,lname=r"\q(PL={:d})".format(PL))

PL=90
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.from_list(col=4,data=vi,lname=r"\q(PL={:d})".format(PL))

PL=100
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.from_list(col=5,data=vi,lname=r"\q(PL={:d})".format(PL))

PL=110
vinf=np.linspace(0,500/3.6,100)
vi=func(PL=PL,vinf=vinf)
op.attach()
wks=op.find_sheet("w","[Book3]Sheet1")
wks.from_list(col=6,data=vi,lname=r"\q(PL={:d})".format(PL))

op.detach()